# 02. Live Telemetry — Attaching an LLM to the Dashboard

A dashboard is only as useful as the events flowing through it. Notebook 01 stood the dashboard up; this one wires a live LLM into it and walks the complete telemetry path — from a single ``on_event`` callback all the way to a browser WebSocket frame, with auth, filtering, and backpressure handled at each hop.

**You will learn:**

- Why Arc separates *event production* (arcllm) from *event delivery* (arcui)
- How ``attach_llm`` wraps an arcllm provider into an ``on_event`` callback
- The exact shape of WebSocket envelopes the dashboard pushes (``event_batch``, ``auth_ok``, ``ping``)
- How first-message auth on ``/ws`` rejects agent tokens and validates viewer/operator roles
- How ``SubscriptionManager`` filters by agent / layer / team server-side
- How ``ConnectionManager`` and ``EventBuffer`` survive a slow browser without dropping the connection
- Where to look in arcllm (notebook 09 — telemetry module) for the events that flow into all of this


## 1. Setup

Run the standard Arc walkthrough boilerplate. It locates the repo root, puts every package's ``src/`` and ``tests/`` directories on ``sys.path``, and loads ``.env`` if one exists. Every notebook in this set starts the same way so cells are interchangeable across packages.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

This notebook is fully **mock-first** — there is no live LLM, no real WebSocket client, and no API keys required. We use Starlette's :class:`TestClient`, which supports synchronous WebSocket interactions over an in-process ASGI transport.

In [ ]:
import json
from unittest.mock import MagicMock

from starlette.testclient import TestClient

from arcui import attach_llm, create_app
from arcui.auth import AuthConfig

print("arcui imports loaded")

## 2. Why live telemetry — operational visibility

Federal deployments care about three things at once: *what happened*, *who did it*, and *can we see it as it happens?* The first two are handled by the audit trail (notebook 09 in arcllm and notebook 04 in arctrust). The third — **live observability** — is what arcui exists for.

The design split is deliberate:

- ``arcllm`` knows how to *produce* telemetry — duration, token counts, cost, errors.   It exposes a single ``on_event`` callback that fires after every call (see notebook 09).
- ``arcui`` knows how to *deliver* telemetry — buffering, filtering, fanout to many browsers,   authentication, backpressure on slow consumers.

Neither side has to know about the other. ``attach_llm`` is the one-line bridge. If you swap the LLM stack tomorrow, the dashboard doesn't change. If you swap the dashboard tomorrow, the LLM doesn't change.

In [ ]:
# A 'why' demo — what an operator sees vs. what the audit log shows.
operator_view = {
    "live": "every call surfaces in <1s",
    "filter": "show only failures, only one agent, only LLM layer",
    "reaction": "cancel a runaway loop before the budget detonates",
}
audit_view = {
    "live": "never (audit is post-hoc)",
    "filter": "searchable by date / agent / event type",
    "reaction": "forensics, compliance evidence",
}
for k in operator_view:
    print(f"{k:10s}  live={operator_view[k]!r:50s}  audit={audit_view[k]!r}")

The two views are complementary. The dashboard is **not** the audit trail — it is the human-readable real-time surface that lets an operator catch problems while there is still time to act. The audit log is what the auditor reads later.

## 3. ``attach_llm`` — wiring an LLM into the dashboard

Read the function once; everything in this notebook follows from these ~20 lines. ``attach_llm`` does three things and nothing else:

1. Builds an ``on_event`` closure that pushes records onto the ``EventBuffer`` and    ingests them into the ``RollingAggregator`` (the windowed stats engine).
2. Walks the LLM's module stack to find ``CircuitBreakerModule``, ``QueueModule``, and    ``TelemetryModule`` instances — these are exposed via ``app.state`` so REST routes    like ``/api/circuit-breakers`` can introspect them without re-walking.
3. Appends the closure to ``app.state.on_event_callbacks`` so the same callback can    be threaded through ``load_model(on_event=...)``.

Critically: the function is synchronous, idempotent for repeated attach calls (each produces an independent callback), and never imports anything from the LLM provider implementation — it only inspects the stack via ``isinstance``.

In [ ]:
app = create_app(auth_config=AuthConfig({"viewer_token": "v", "operator_token": "o"}))

# A minimal stand-in for an LLM provider. The real shape is an arcllm
# adapter + module stack chained via _inner. Here we use a bare MagicMock
# so attach_llm just exercises the on_event wiring.
fake_llm = MagicMock()
fake_llm._inner = None  # end of stack

attach_llm(app, fake_llm, label="analyst-1")

print("callbacks attached:", len(app.state.on_event_callbacks))
print("circuit breakers found:", len(app.state.circuit_breakers))
print("telemetry modules found:", len(app.state.telemetry_modules))
print("queue modules found:", len(app.state.queue_modules))

The `label` argument is stamped onto every event the callback emits so the dashboard can attribute it to a specific logical agent. If you attach the same physical LLM twice with different labels (e.g., once for ``analyst-1``, once for ``planner-2``), every event is duplicated — once per attached callback — which is exactly what you want when the same model serves multiple agents.

In [ ]:
# Fire one fake telemetry record through the callback we just attached.
callback = app.state.on_event_callbacks[0]

record = MagicMock()
record.model_dump.return_value = {
    "total_tokens": 412,
    "input_tokens": 312,
    "output_tokens": 100,
    "cost_usd": 0.00318,
    "duration_ms": 1184.7,
    "model": "claude-sonnet-4-6",
    "provider": "anthropic",
}

callback(record)

print("events buffered:", app.state.event_buffer.pending_count)
stats = app.state.aggregator.stats("1h")
print("aggregator request_count:", stats["request_count"])
print("aggregator total_cost_usd:", round(stats.get("total_cost_usd", 0.0), 5))

Two side effects from one callback invocation: the buffer holds the dict for the next flush, and the aggregator's 1-hour window has been bumped. The buffer feeds the WebSocket; the aggregator feeds REST endpoints (``/api/stats``, ``/api/cost-efficiency``).

In [ ]:
# What gets stamped onto the record? agent_label is set if the dict
# doesn't already carry one. Existing labels are NEVER overwritten —
# this lets a multi-agent setup with its own labelling stay authoritative.
rec_with_label = MagicMock()
rec_with_label.model_dump.return_value = {
    "agent_label": "authoritative-source",
    "total_tokens": 100,
    "cost_usd": 0.001,
    "duration_ms": 50.0,
}

callback(rec_with_label)
# Inspect the buffer's tail directly to confirm the label was preserved.
buffered = list(app.state.event_buffer._buffer)[-1]
print("preserved label:", buffered["agent_label"])

An attached LLM with a stacked ``CircuitBreakerModule`` shows the discovery in action. We construct a tiny synthetic stack — adapter wrapped by a circuit breaker wrapped by a telemetry module — and attach it. The discovery walk records both modules, leaving the actual adapter alone because nothing in arcui needs to know about it.

In [ ]:
from arcllm.modules.circuit_breaker import CircuitBreakerModule
from arcllm.modules.telemetry import TelemetryModule

fresh_app = create_app()

adapter = MagicMock()
adapter._inner = None

cb = MagicMock(spec=CircuitBreakerModule)
cb._inner = adapter

outer = MagicMock(spec=TelemetryModule)
outer._inner = cb

attach_llm(fresh_app, outer, label="full-stack-agent")

print("circuit_breakers:", len(fresh_app.state.circuit_breakers))
print("telemetry_modules:", len(fresh_app.state.telemetry_modules))

The walk uses ``getattr(current, '_inner', None)`` so a stack that ends in a bare adapter (no ``_inner`` attribute on the leaf) terminates cleanly without raising. This is also why ``attach_llm`` works with a plain ``MagicMock`` — as long as ``_inner`` returns ``None`` eventually, the walk terminates.

## 4. WebSocket protocol — message envelopes, event types

The dashboard speaks one protocol on ``/ws``. It is intentionally tiny:

**Client → server**

- ``{"token": "<bearer>"}`` — first message, must arrive within 5 s.
- ``{"type": "subscribe", "agents": [...], "layers": [...], "teams": [...]}`` — server-side filter.
- ``{"type": "subscribe:agent", "agent_id": "<id>"}`` — open per-agent file watcher.
- ``{"type": "unsubscribe:agent", "agent_id": "<id>"}`` — close it.

**Server → client**

- ``{"type": "auth_ok", "role": "viewer|operator"}`` — auth accepted.
- ``{"error": "..."}`` — auth failed (followed by a close).
- ``{"type": "ping"}`` — keepalive every 30 s.
- ``{"type": "event_batch", "events": [...]}`` — telemetry from ``attach_llm``.
- ``UIEvent`` envelopes (flat) — telemetry from the multi-agent reporter / bridge.

Every message is JSON. There is no binary path, no compression layer — the protocol is boring on purpose: easy to debug with ``websocat``, easy to mock in tests, easy to log.

In [ ]:
# Stand up an app and walk the full handshake using TestClient.
auth = AuthConfig({"viewer_token": "walk-viewer", "operator_token": "walk-op"})
app2 = create_app(auth_config=auth)
client = TestClient(app2)

with client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    auth_resp = ws.receive_json()
    print("first server frame:", auth_resp)

Send a server-side broadcast and read the next frame on the client. The TestClient round-trips through the real Starlette ASGI machinery, so what we observe here is byte-for-byte what a browser would receive.

In [ ]:
with client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    ws.receive_json()  # auth_ok

    app2.state.connection_manager.broadcast(
        {
            "type": "event_batch",
            "events": [
                {"trace_id": "tr-001", "cost_usd": 0.0042, "model": "claude-sonnet-4-6"},
                {"trace_id": "tr-002", "cost_usd": 0.0011, "model": "claude-sonnet-4-6"},
            ],
        }
    )
    payload = ws.receive_json()
    print("envelope type:", payload["type"])
    print("events delivered:", len(payload["events"]))
    print("first event:", payload["events"][0])

An ``event_batch`` is a single WebSocket frame containing many TraceRecord-shaped dicts. Batching is what makes the dashboard cheap at scale: a fleet of 1000 agents each emitting one event per second produces 1000 events/s, but the browser receives ~10 frames/s because the ``EventBuffer`` flushes every 100 ms.

In [ ]:
# Show the connection-manager's queue-fanout. Each client gets its own asyncio.Queue.
import asyncio

from arcui.connection import ConnectionManager

demo_cm = ConnectionManager()
q1 = demo_cm.create_queue()
q2 = demo_cm.create_queue()
demo_cm.broadcast({"frame": 1})
demo_cm.broadcast({"frame": 2})


# Queue.empty() and get_nowait() are synchronous — no event loop needed.
def drain(q):
    out = []
    while not q.empty():
        out.append(json.loads(q.get_nowait()))
    return out


print("q1 received:", drain(q1))
print("q2 received:", drain(q2))

Both queues received both frames — that is the broadcast guarantee. If a queue is full the **oldest** message is dropped to make room (see ``ws_helpers.safe_enqueue``); a slow client cannot back up the broadcast loop. We exercise that in section 7.

## 5. Authentication on WS connections

The WebSocket auth is intentionally **not** the same as the REST auth. REST runs through ``AuthMiddleware`` and reads ``Authorization: Bearer <token>`` headers. WebSockets accept the connection first, then expect a JSON message containing the token within 5 seconds. If the message is malformed, missing, or the token is wrong, the server sends an error frame and closes the connection with code 4003.

Three rules govern the WS auth path:

1. **Auth before any data.** No event reaches an unauthenticated client.
2. **Agent tokens are rejected.** ``role == "agent"`` triggers an immediate close —    agent tokens are for ``/api/agent/connect``, never for ``/ws``. This is OWASP    ASI-03 (Identity & Privilege Abuse) hardening: an agent can push events but must not    read other agents' events.
3. **Constant-time comparison.** Token validation uses ``hmac.compare_digest`` to    eliminate timing side channels.

In [ ]:
# A bad token is rejected and the connection is closed.
with client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "definitely-not-the-token"}))
    err = ws.receive_json()
    print("error frame:", err)

The same shape is used for malformed JSON, missing token field, and timeout — the client always gets a single ``{"error": ...}`` frame, then a normal WebSocket close. There is no distinction in the close code between the failure modes; that information stays in the audit log to avoid telling a probe exactly *why* their attempt failed.

In [ ]:
# An operator token works the same way as a viewer, with a different role label.
with client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "walk-op"}))
    resp = ws.receive_json()
    print("role:", resp["role"])
    assert resp["type"] == "auth_ok"

The role on ``auth_ok`` is informational — the server enforces role-based authorization on the routes that need it (e.g., admin endpoints check for ``role == 'operator'``). Both viewer and operator can read the live event stream; only operator can issue control messages on the ``/api/agent`` socket.

In [ ]:
# Agent tokens are rejected on /ws even though they're valid for /api/agent/connect.
auth_with_agent = AuthConfig(
    {
        "viewer_token": "v",
        "operator_token": "o",
        "agent_token": "walk-agent",
    }
)
agent_app = create_app(auth_config=auth_with_agent)
agent_client = TestClient(agent_app)

with agent_client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "walk-agent"}))
    err = ws.receive_json()
    print("agent token rejected:", err)

Notice the close code is 4003 — same as a fully-invalid token. The server makes no distinction; an attacker who somehow obtains a leaked agent token still cannot read the live stream and cannot tell from the response whether the token *would* work elsewhere. From the WS endpoint's point of view the answer is simply ``no``.

## 6. Event filtering — server-side and client-side

Once authenticated, a browser can **narrow** the firehose with a ``subscribe`` message. Filtering happens **server-side** so the bandwidth cost is paid once regardless of how many tabs are open. The filter object is a ``Subscription`` model with three optional list fields:

- ``agents`` — list of agent ids; ``None`` = all agents
- ``layers`` — list of ``llm`` / ``run`` / ``agent`` / ``team`` / ``scheduler``; ``None`` = all
- ``teams`` — list of team names; ``None`` = all

The match is *AND*: a UIEvent must satisfy every set field. ``None`` on a field means the field doesn't constrain. A queue without a registered subscription is **fail-open** — it matches every event so a client that connects before its filter is registered does not silently miss data.

In [ ]:
from arcui.subscription import Subscription, SubscriptionManager
from arcui.types import UIEvent

sub_mgr = SubscriptionManager()
q = asyncio.Queue()
sub_mgr.set_subscription(q, Subscription(agents=["analyst-1"], layers=["llm"]))

# An event that matches both filters
match = UIEvent(
    layer="llm",
    event_type="request_complete",
    agent_id="analyst-1",
    agent_name="analyst",
    source_id="analyst-1",
    timestamp="2026-05-08T12:00:00Z",
    data={"cost_usd": 0.001},
    sequence=1,
)

# Same event but for a different agent — agents filter rejects it
wrong_agent = match.model_copy(update={"agent_id": "planner-2"})

# Same event but on the run layer — layers filter rejects it
wrong_layer = match.model_copy(update={"layer": "run"})

for ev, label in [(match, "match"), (wrong_agent, "wrong agent"), (wrong_layer, "wrong layer")]:
    print(f"{label:14s} -> matches={sub_mgr.matches(q, ev)}")

The actual delivery path uses ``broadcast_filtered``, which serialises the event exactly once and ``safe_enqueue``s the JSON string into every queue whose subscription matches. Serialisation is the expensive step; doing it once and broadcasting the string avoids paying it per-client.

In [ ]:
# Broadcast through the manager and observe per-queue delivery.
filtering_mgr = SubscriptionManager()
qa = asyncio.Queue()
qb = asyncio.Queue()
filtering_mgr.set_subscription(qa, Subscription(layers=["llm"]))
filtering_mgr.set_subscription(qb, Subscription(layers=["run"]))

ev = UIEvent(
    layer="llm",
    event_type="request_complete",
    agent_id="a1",
    agent_name="a1",
    source_id="a1",
    timestamp="2026-05-08T12:00:01Z",
    data={},
    sequence=2,
)
filtering_mgr.broadcast_filtered(ev)

print("qa size (llm subscriber):", qa.qsize())
print("qb size (run subscriber):", qb.qsize())

Only ``qa`` got the event — the ``run`` subscriber sees an empty queue because the filter rejected the ``llm`` envelope before it ever entered the queue. This is the **server-side** filter. The browser may also do **client-side** filtering on top (e.g., a UI toggle for ``show only failures``) but that is purely UI logic; nothing stops the browser from displaying everything it receives.

In [ ]:
# Sending a subscribe message over the live WS path. We exercise the same
# code path that the dashboard JS uses on first connect.
with client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    ws.receive_json()  # auth_ok

    ws.send_text(
        json.dumps(
            {
                "type": "subscribe",
                "agents": ["analyst-1"],
                "layers": ["llm"],
            }
        )
    )

    # No ack frame — subscribe is fire-and-forget. The server logs the
    # update, audit-events it, and applies the new filter to subsequent
    # broadcasts. We confirm by sending a non-matching event and observing
    # that nothing arrives without blocking forever.
    app2.state.connection_manager.broadcast({"type": "unfiltered_passthrough"})
    drained = ws.receive_json()  # raw broadcast still passes (dict, not UIEvent)
    print("raw passthrough:", drained)

A subtle point: the ``subscribe`` filter only applies to **UIEvent** broadcasts that go through ``SubscriptionManager.broadcast_filtered``. The ``ConnectionManager.broadcast`` path used for raw dicts (the ``attach_llm`` ``event_batch`` envelope) is intentionally unfiltered server-side — the browser does its own client-side filtering on those because they don't carry a typed ``layer`` / ``agent_id`` shape the server can statically reason about.

## 7. Backpressure handling on slow clients

A real dashboard has at least one tab open on a laptop in a coffee shop with a flaky connection. If the browser stalls for two seconds, where do the events go?

The answer is in three places:

1. **Per-client bounded queue.** Every WebSocket client gets its own ``asyncio.Queue``    with ``maxsize=1000`` (about 8 KB of typical JSON). The broadcast loop never blocks    on a full queue; it drops the **oldest** message and inserts the new one.
2. **EventBuffer flush cadence.** Events accumulate in a ``deque(maxlen=1000)`` and    flush every 100 ms. Even if every WS client is slow, the producer side never stalls    — it can drop into the bounded deque indefinitely.
3. **Heartbeat ping.** A 30-second ``{'type': 'ping'}`` frame keeps idle proxies open    and surfaces dead connections so the cleanup path runs.

The combined invariant: a slow or dead browser **never** stalls the producer. Worst case, a stalled client misses some events; it never blocks anyone else.

In [ ]:
from arcui.ws_helpers import safe_enqueue

# A queue with maxsize=3 simulates a tiny per-client buffer.
tiny = asyncio.Queue(maxsize=3)
for i in range(5):
    safe_enqueue(tiny, f"msg-{i}")

# Drain to see what survived.
drained = []
while not tiny.empty():
    drained.append(tiny.get_nowait())
print("survivors after overflow:", drained)

The first two messages were evicted; the **most recent** three made it through. For a telemetry feed this is the right tradeoff: an operator who steps away and comes back wants to see what's happening *now*, not the oldest backlog of stale events.

In [ ]:
# EventBuffer flush behaviour: even with no clients, push() never raises.
from arcui.event_buffer import EventBuffer

isolated_cm = ConnectionManager()
buf = EventBuffer(isolated_cm, maxlen=4)
for i in range(8):
    buf.push({"i": i})
print("buffer pending:", buf.pending_count, "(maxlen=4 caps it)")

buf.flush_once()  # flushes when there are clients; otherwise no-op
print("after flush:", buf.pending_count)

When there are no connected clients, ``flush_once`` is a no-op — events stay in the buffer until a client arrives or the bounded ``maxlen`` overwrites them. This is what lets ``attach_llm`` run during dashboard startup before the browser has connected: the first 100 ms of telemetry isn't lost, it's queued.

In [ ]:
# Heartbeat: every 30 seconds the server sends {type: ping}. The client may
# answer with anything (a pong, a no-op). Browsers send a TCP-level pong
# automatically; arcui's frontend treats the ping as a 'connection alive'
# signal. Here we just observe one cycle by reading the constant.
from arcui.ws_helpers import HEARTBEAT_INTERVAL_SECONDS

print(f"heartbeat interval: {HEARTBEAT_INTERVAL_SECONDS}s")
print("frame shape:", {"type": "ping"})

A ``ping`` arriving in the middle of an ``event_batch`` stream is normal traffic — the client filters by ``type`` and ignores it. The server doesn't expect a pong in JSON; the WebSocket protocol's TCP-level keepalive is what actually verifies liveness, and the JSON ping is for any HTTP/2 proxy that would otherwise close an idle stream.

In [ ]:
# DoS guard: a giant client message is dropped silently — connection stays alive.
from arcui.ws_helpers import MAX_WS_MESSAGE_SIZE

with client.websocket_connect("/ws") as ws:
    ws.send_text(json.dumps({"token": "walk-viewer"}))
    ws.receive_json()  # auth_ok

    oversized = "x" * (MAX_WS_MESSAGE_SIZE + 1)
    ws.send_text(oversized)

    # Connection is still alive — broadcast and confirm.
    app2.state.connection_manager.broadcast({"type": "still_alive"})
    payload = ws.receive_json()
    print("after oversized:", payload)

An oversized message — 1 MB + 1 byte or larger — is **silently skipped**. Disconnecting would let a malicious client trivially knock real operators off the dashboard by spamming oversized frames; logging-and-continuing is the right balance. The ``MAX_WS_MESSAGE_SIZE`` constant is shared with the agent WebSocket so both endpoints behave identically.

## 8. Cross-ref to arcllm telemetry module (notebook 09) for source events

Everything in this notebook starts from a single function: ``arcllm.modules.telemetry.TelemetryModule``. That module is what generates the ``TraceRecord`` objects that ``attach_llm`` then forwards. The contract between the two packages is exactly:

```python
from arcllm import load_model
from arcui import attach_llm, create_app

app = create_app()
model = load_model('anthropic', telemetry=True, on_event=...)  # arcllm produces
attach_llm(app, model, label='analyst')                         # arcui delivers
```

The arcllm side (covered in notebook ``arcllm/09-telemetry-module.ipynb``) handles:

- Phase sub-timings (queue wait, network, deserialise, total)
- Token / cost accounting from response usage
- Provider / model / agent_label attribution
- Threading the ``on_event`` callback through every layer of the module stack

The arcui side (this notebook) handles:

- Buffering and batching for WebSocket fanout
- Per-client filtering and authentication
- Backpressure / DoS resistance
- Aggregator windowing for REST stats

The split is what makes both packages testable in isolation. arcllm tests don't need Starlette; arcui tests don't need an HTTP-mocked LLM provider.

In [ ]:
# A demonstration of the contract — we import both ends and confirm the symbols exist.
import arcllm
from arcllm.modules import telemetry as arcllm_telemetry
import arcui

print("arcllm version:", getattr(arcllm, "__version__", "unknown"))
print("arcui version:", arcui.__version__)
print("arcllm TelemetryModule available:", hasattr(arcllm_telemetry, "TelemetryModule"))
print("arcui attach_llm available:", hasattr(arcui, "attach_llm"))

If you want to *modify* the shape of the events flowing through the dashboard, the right place to look is the arcllm side — change what ``TelemetryModule`` emits and arcui will faithfully relay it. If you want to change *how* events are filtered, buffered, or authenticated, that's arcui's job and lives entirely in the modules we exercised in sections 4–7.

In [ ]:
# A quick reference: the four arcui modules the contract leans on.
from arcui import (
    attach_llm,
    create_app,
    serve,
)
from arcui.aggregator import RollingAggregator
from arcui.connection import ConnectionManager
from arcui.event_buffer import EventBuffer
from arcui.subscription import Subscription, SubscriptionManager

for sym in [
    attach_llm,
    create_app,
    serve,
    RollingAggregator,
    ConnectionManager,
    EventBuffer,
    Subscription,
    SubscriptionManager,
]:
    print(f"{sym.__name__:24s} from {sym.__module__}")

These eight symbols are the entire surface area you need to wire telemetry into the dashboard. Three are public re-exports (``attach_llm``, ``create_app``, ``serve``) and five are the internal modules ``attach_llm`` composes for you.

In [ ]:
# Show the warm-start bridge: when create_app receives a trace_store, serve() warms
# the aggregator from historical data before opening the WS port. This means the
# first browser to connect after a restart sees a non-empty stats panel instantly.
store = MagicMock()
store.iter_records = lambda: _aiter([])  # empty trace history
warm_app = create_app(trace_store=store)
print("trace_store wired:", warm_app.state.trace_store is store)

# We do not actually run warm_start() here because TestClient handles the lifespan.
# The warm_start() call lives inside serve() and uses the same merge_by_timestamp
# helper that aggregator.py uses for the live ingest path — one merge implementation
# for both warm and live data.

Helper for the cell above:

In [ ]:
async def _aiter(items):
    for x in items:
        yield x


print("async helper defined")

If you want to drive the warm-start path end-to-end, see notebook ``arcllm/14-trace-store.ipynb`` — it constructs a ``JSONLTraceStore``, writes records, and shows how arcui consumes them. The two notebooks plug together cleanly: arcllm/14 produces the trace files, and this notebook (combined with arcui/01) shows how they bootstrap the dashboard.

## 9. Summary

- **Why live telemetry exists.** Audit logs answer *what happened*; the dashboard   answers *what is happening*, in time for an operator to act.
- **``attach_llm`` is the bridge.** One function call wires an arcllm provider into   the dashboard's ``EventBuffer`` and ``RollingAggregator`` and discovers stack modules   (circuit breaker, telemetry, queue) so REST endpoints can introspect them.
- **The WebSocket protocol is intentionally tiny.** First-message JSON auth, then a   small set of typed envelopes (``auth_ok``, ``ping``, ``event_batch``, ``UIEvent``).
- **Auth is fail-closed.** Missing/invalid/agent tokens get one error frame and a 4003   close. Constant-time comparison defends against timing side channels.
- **Filtering is server-side.** ``Subscription`` narrows the firehose by agent / layer /   team; ``broadcast_filtered`` serialises once and fans out to matching queues.
- **Backpressure is built in.** Bounded per-client queues drop the oldest, ``EventBuffer``   flushes every 100 ms with its own bounded deque, and 1 MB+ client frames are silently   skipped without dropping the connection.
- **Cross-package contract.** arcllm produces telemetry; arcui delivers it. Either side   can be replaced without touching the other.

**Next:** see ``arcui/01-dashboard-bringup.ipynb`` for the matching three-token-auth and dashboard launch story (this notebook assumed an app was already created); see ``arcllm/09-telemetry-module.ipynb`` for the source-side event production.

**API surface covered:**

- ``arcui.create_app``, ``arcui.attach_llm``, ``arcui.serve``
- ``arcui.auth.AuthConfig``
- ``arcui.connection.ConnectionManager``
- ``arcui.event_buffer.EventBuffer``
- ``arcui.subscription.Subscription``, ``arcui.subscription.SubscriptionManager``
- ``arcui.aggregator.RollingAggregator``
- ``arcui.ws_helpers.safe_enqueue``, ``MAX_WS_MESSAGE_SIZE``, ``HEARTBEAT_INTERVAL_SECONDS``
- ``arcui.types.UIEvent``
- WebSocket protocol: ``token`` first-frame auth, ``auth_ok`` / ``error`` / ``ping`` /   ``event_batch`` envelopes, ``subscribe`` / ``subscribe:agent`` / ``unsubscribe:agent`` messages.